## Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

## Read bronze table

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")

## Silver Transformation

### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Clean up customer id

In [0]:
df = (
    df.withColumn(
        "CID",
        F.when(col("CID").startswith("NAS"),
               F.substring(col("CID"), 4, F.length(col("CID"))))
        .otherwise(col("CID"))
    )
)

### Birthdate validation

In [0]:
df = (
    df.withColumn(
        "BDATE",
        F.when(col("BDATE") > F.current_date(), None)
        .otherwise(col("BDATE"))
    )
)

### Gender normalization

In [0]:
df = (
    df.withColumn(
        "GEN",
        F.when(F.upper(trim(col("GEN"))).isin("F", "FEMALE"), "Female")
        .when(F.upper(trim(col("GEN"))).isin("M", "MALE"), "Male")
        .otherwise("n/a")
    )
)

### Rename columns

In [0]:
RENAME_MAP = {
    "CID": "customer_id",
    "BDATE": "birthdate",
    "GEN": "gender"
}

for old, new in RENAME_MAP.items():
    df = df.withColumnRenamed(old, new)

### Sanity check of column

In [0]:
df.limit(30).display()

customer_id,birthdate,gender
AW00011000,1971-10-06,Male
AW00011001,1976-05-10,Male
AW00011002,1971-02-09,Male
AW00011003,1973-08-14,Female
AW00011004,1979-08-05,Female
AW00011005,1976-08-01,Male
AW00011006,1976-12-02,Female
AW00011007,1969-11-06,Male
AW00011008,1975-07-04,Female
AW00011009,1969-09-29,Male


### Write to silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")


### Sanity check of silver table

In [0]:
%sql
SELECT *
FROM workspace.silver.erp_customers
LIMIT 20

customer_id,birthdate,gender
AW00011000,1971-10-06,Male
AW00011001,1976-05-10,Male
AW00011002,1971-02-09,Male
AW00011003,1973-08-14,Female
AW00011004,1979-08-05,Female
AW00011005,1976-08-01,Male
AW00011006,1976-12-02,Female
AW00011007,1969-11-06,Male
AW00011008,1975-07-04,Female
AW00011009,1969-09-29,Male
